# Create Arthritis Foundation Awards

Creates Arthritis Foundation (US) awards from the structured awardee lists in
AF's own funding announcements on arthritis.org (AF publishes no grants
database or export). `scripts/local/arthritis_foundation_to_s3.py` reads a fixed
set of 12 announcement pages and parses only their awardee lists (investigator,
institution, programme, amount where given, project title); prose-only
announcements are not used. **85 grants shipped, 2018-2026**: RA Research
Program 2023-2025, FastOA Hip Cohorts 2024, CARRA-Arthritis Foundation Grant
Program 2018 / 2023-2026 and Childhood Research Grants 2018-2019 (joint AF/CARRA
programme, AF-funded), AF/AOFAS Ankle Arthritis Think Tank 2023 (co-funded with
AOFAS). 100% title / PI / institution; amount 36% (the CARRA and AOFAS lists
publish no per-grant amount; §6.7 amount check waived).

**`funder_award_id`:** no grant number is published on these pages; synthetic
`AF-{year}-{programme code}-{LEAD FAMILY NAME}-{given initial}`. Collapses onto 0
existing citation stubs.

**Overlap with AF's own Crossref grant deposits.** AF registers some grant DOIs
itself (prefix 10.57104, ProposalCentral award numbers; 33 deposits, mainly RA
Research Program and OA grants), already in `openalex_awards_raw` as
`crossref_work` rows (priority 1). The script matches each parsed grant to those
deposits (lead family name + first initial, title token overlap >= 0.5) and does
**not** ship the 8 matches (`excluded = '1'`, `exclusion_reason =
'covered_by_crossref_grant'`, deposit number in `crossref_award`). This notebook
ships only `excluded = '0'` rows.

**Twin:** Arthritis Research Initiative (F4320307130) shares AF's ROR 04zk9fe75
(10 shell rows only); this ingest ships under the tracker's F4320306237.

**Dates:** `start_year` = announcement year; `start_date` = Jan 1 of that year
(§2.3 year-only pattern). No end dates published.

**Prerequisites:** run `scripts/local/arthritis_foundation_to_s3.py` first. It
uploads `s3://openalex-ingest/awards/arthritis_foundation/arthritis_foundation_projects.parquet`
(§1.4 shrink guard).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306237`
- display_name: Arthritis Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 04zk9fe75, doi 10.13039/100000980)

**Priority:** `538` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.arthritis_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/arthritis_foundation/arthritis_foundation_projects.parquet`;

In [ ]:
%sql
SELECT programme, year, excluded, COUNT(*) as n
FROM openalex.awards.arthritis_foundation_raw GROUP BY 1, 2, 3 ORDER BY 1, 2, 3;

In [ ]:
%sql
DESCRIBE openalex.awards.arthritis_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.arthritis_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320306237 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320306237;

## Step 2: Create Arthritis Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.arthritis_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306237  -- Arthritis Foundation
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) END as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.scheme RLIKE '(?i)fellow' THEN 'fellowship' ELSE 'research' END as funding_type,
    NULLIF(TRIM(g.scheme), '') as funder_scheme,
    'arthritis_foundation_announcements' as provenance,
    TRY_TO_DATE(CONCAT(g.year, '-01-01'), 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw.
    named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) as lead_investigator,
    CASE WHEN NULLIF(TRIM(g.co_lead_family_name), '') IS NOT NULL THEN
    named_struct(
        'given_name', NULLIF(TRIM(g.co_lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.co_lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.co_lead_institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.arthritis_foundation_raw g
CROSS JOIN src_funder f
WHERE g.excluded = '0'
  AND g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
  AND NULLIF(TRIM(g.title), '') IS NOT NULL;

In [ ]:
%sql
-- Shape check: every funder_award_id is the synthetic AF-<year>-<code>-<FAMILY>-<I>; expect 0 rows.
SELECT funder_award_id FROM openalex.awards.arthritis_foundation_awards
WHERE NOT funder_award_id RLIKE '^AF-(19|20)[0-9]{2}-[A-Z]+-[A-Z]+-[A-Z](-[0-9]+)?$'
LIMIT 20;

In [ ]:
%sql
-- Must be 0: no shipped grant duplicates one of AF's own crossref_work grant records
-- (same lead investigator and the same title).
SELECT a.funder_award_id, a.display_name, c.funder_award_id as crossref_award
FROM openalex.awards.arthritis_foundation_awards a
JOIN openalex.awards.openalex_awards_raw c
  ON c.funder_id = 4320306237 AND c.provenance = 'crossref_work'
 AND LOWER(c.lead_investigator.family_name) = LOWER(a.lead_investigator.family_name)
 AND LOWER(REGEXP_REPLACE(c.display_name, '[^A-Za-z0-9]', '')) = LOWER(REGEXP_REPLACE(a.display_name, '[^A-Za-z0-9]', ''));

In [ ]:
%sql
-- Never shrink what may already be published: refuse to replace a larger earlier load.
SELECT assert_true(
    (SELECT COUNT(*) FROM openalex.awards.arthritis_foundation_awards) >=
    (SELECT COUNT(*) FROM openalex.awards.openalex_awards_raw WHERE provenance = 'arthritis_foundation_announcements' AND priority = 538),
    'arthritis_foundation_awards is smaller than the rows already in openalex_awards_raw; refusing to shrink'
) as no_shrink;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'arthritis_foundation_announcements' AND priority = 538;

-- Insert into openalex_awards_raw with priority.
-- Priority 538: direct-from-funder ingest of AF's award announcements.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    538 as priority
FROM openalex.awards.arthritis_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_af_awards, COUNT(DISTINCT id) as distinct_ids FROM openalex.awards.arthritis_foundation_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name,
       co_lead_investigator.family_name as co_lead
FROM openalex.awards.arthritis_foundation_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.arthritis_foundation_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.arthritis_foundation_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.arthritis_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.arthritis_foundation_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect 100% title/PI/institution, ~36% amount by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(co_lead_investigator) as has_co_lead
FROM openalex.awards.arthritis_foundation_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'arthritis_foundation_announcements'
GROUP BY provenance, priority;